# Steinbeck, Akerlof, and the Market for Jalopies  
## A full research notebook with interactive sliders

This notebook reconstructs a research-grade agent-based model for the used-car market described in *The Grapes of Wrath* and interpreted through, but not reduced to, the Akerlof lemons mechanism.

The underlying theoretical idea is the following.

A standard Akerlof environment explains market failure through **quality heterogeneity** and **asymmetric information**. Buyers cannot perfectly observe quality before purchase, so the market price reflects expected average quality. As a consequence, higher-quality sellers may exit and lower-quality goods may dominate.

The Steinbeck case is richer. In the literary-historical environment described by Steinbeck, market failure is amplified by:

1. **forced demand**: buyers urgently need transport and cannot easily postpone purchase;
2. **low market literacy**: buyers are poorly informed not only about quality, but also about the rules and tactics of the market;
3. **local market power**: sellers face vulnerable, transient buyers in a context close to a one-shot market;
4. **opportunism and price discrimination**: sellers exploit urgency and vulnerability;
5. **weak corrective institutions**: there is no effective minimum-quality enforcement or credible redress.

This notebook is designed to:

- let you run a **baseline** and a set of **comparative scenarios**;
- let you change parameters interactively through **ipywidgets sliders**;
- save **figures** into a `figures/` folder;
- save **tables** into a `tables/` folder, including LaTeX-ready tables for the paper;
- provide a transparent computational object that can be discussed in a paper and replicated by others.

**Time unit:** one period = **one week**.  
A typical one-year simulation therefore uses **52 periods**.

In [ ]:
# Core imports

import os
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, Markdown

BASE_DIR = Path(".")
FIG_DIR = BASE_DIR / "figures"
TAB_DIR = BASE_DIR / "tables"

FIG_DIR.mkdir(exist_ok=True)
TAB_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## 1. Parameterization

The model is deliberately kept simple enough to remain transparent, but rich enough to separate distinct mechanisms.

### Main interpretation of parameters

- `n_buyers`: number of buyers arriving in the market each week;
- `n_sellers`: number of sellers active in the market each week;
- `periods`: number of weeks simulated;
- `cars_per_seller`: average number of cars each seller brings to market;
- `share_good`: ex ante share of potentially good cars in the population of used cars;
- `info_asymmetry`: noise in buyers' perception of quality;
- `urgency_mean`: average urgency of purchase;
- `literacy_mean`: average market literacy of buyers;
- `seller_opportunism`: degree to which sellers try to exploit lemons strategically;
- `price_discrimination`: degree to which prices are raised for more vulnerable buyers;
- `institution_strength`: strength of inspections / minimum-quality enforcement;
- `competition_strength`: degree of effective seller competition (higher means less power to overcharge).

In [ ]:
@dataclass
class Params:
    # Market size and time
    n_buyers: int = 600
    n_sellers: int = 60
    periods: int = 52
    cars_per_seller: int = 4

    # Car quality structure
    share_good: float = 0.35
    good_quality_low: float = 0.62
    good_quality_high: float = 0.95
    lemon_quality_low: float = 0.08
    lemon_quality_high: float = 0.48

    # Buyer side
    buyer_budget_low: float = 60.0
    buyer_budget_high: float = 220.0
    urgency_mean: float = 0.72
    urgency_sd: float = 0.18
    literacy_mean: float = 0.38
    literacy_sd: float = 0.18
    risk_tolerance_mean: float = 0.55
    risk_tolerance_sd: float = 0.18
    vulnerability_weight: float = 0.60

    # Information and pricing
    info_asymmetry: float = 0.42
    base_markup: float = 0.18
    seller_opportunism: float = 0.55
    price_discrimination: float = 0.55
    competition_strength: float = 0.25

    # Institutions
    institution_strength: float = 0.12
    minimum_quality_threshold: float = 0.16
    fine_multiplier: float = 0.40

    # Stochastic control
    random_seed: int = 7

    @property
    def effective_cars_per_seller(self):
        return max(1, int(round(self.cars_per_seller)))

DEFAULT_PARAMS = Params()
DEFAULT_PARAMS

## 2. Agents and goods

The model contains three object types.

### Cars
A car has:
- a true quality in `[0, 1]`;
- a cost for the seller;
- a label `good` or `lemon`.

### Buyers
A buyer has:
- a budget;
- urgency;
- market literacy;
- risk tolerance;
- a synthetic vulnerability score.

### Sellers
A seller holds an inventory of cars and posts prices that depend on:
- the car's quality and cost,
- the buyer's vulnerability,
- the scope for price discrimination,
- opportunism,
- the competitive and institutional environment.

In [ ]:
class Car:
    def __init__(self, quality, cost):
        self.quality = float(quality)
        self.cost = float(cost)
        self.is_lemon = self.quality < 0.5

    @classmethod
    def draw(cls, p: Params, rng: np.random.Generator):
        if rng.random() < p.share_good:
            quality = rng.uniform(p.good_quality_low, p.good_quality_high)
        else:
            quality = rng.uniform(p.lemon_quality_low, p.lemon_quality_high)
        # Cost is increasing in quality, with some random variation
        cost = 18 + 85 * quality + rng.normal(0, 4)
        cost = max(8.0, cost)
        return cls(quality=quality, cost=cost)


class Buyer:
    def __init__(self, p: Params, rng: np.random.Generator):
        self.budget = float(rng.uniform(p.buyer_budget_low, p.buyer_budget_high))
        self.urgency = float(np.clip(rng.normal(p.urgency_mean, p.urgency_sd), 0, 1))
        self.literacy = float(np.clip(rng.normal(p.literacy_mean, p.literacy_sd), 0, 1))
        self.risk_tolerance = float(np.clip(rng.normal(p.risk_tolerance_mean, p.risk_tolerance_sd), 0, 1))

        # Vulnerability rises when literacy is low and urgency is high
        vulnerability = p.vulnerability_weight * (1 - self.literacy) + (1 - p.vulnerability_weight) * self.urgency
        self.vulnerability = float(np.clip(vulnerability, 0, 1))

    def perceived_quality(self, car: Car, p: Params, rng: np.random.Generator):
        # More info asymmetry -> noisier perception
        # More literacy -> smaller effective noise
        effective_noise = p.info_asymmetry * (1 - 0.65 * self.literacy)
        noisy_signal = car.quality + rng.normal(0, effective_noise)
        return float(np.clip(noisy_signal, 0, 1))

    def reservation_price(self, perceived_quality: float):
        # Willingness to pay increases with perceived quality and urgency,
        # and slightly with risk tolerance.
        urgency_factor = 1 + 0.60 * self.urgency
        risk_factor = 1 + 0.15 * self.risk_tolerance
        return self.budget * perceived_quality * urgency_factor * risk_factor


class Seller:
    def __init__(self, p: Params, rng: np.random.Generator):
        self.inventory = [Car.draw(p, rng) for _ in range(p.effective_cars_per_seller)]
        self.rng = rng

    def quote_price(self, buyer: Buyer, car: Car, p: Params):
        # Base margin on cost
        base = car.cost * (1 + p.base_markup)

        # Vulnerability-based price discrimination
        discrim = 1 + p.price_discrimination * buyer.vulnerability

        # Opportunism increases especially on lemons
        lemon_component = p.seller_opportunism * (1.0 if car.is_lemon else 0.25)

        # Competition weakens the ability to exploit
        competition_discount = 1 - 0.55 * p.competition_strength

        quoted = base * (discrim + lemon_component) * competition_discount
        return float(max(car.cost, quoted))

    def best_offer_for_buyer(self, buyer: Buyer, p: Params):
        # Seller chooses the car that maximizes expected extractable surplus
        # from this particular buyer, which is consistent with targeted selling.
        best_tuple = None
        for idx, car in enumerate(self.inventory):
            price = self.quote_price(buyer, car, p)
            score = price - car.cost
            candidate = (score, idx, car, price)
            if best_tuple is None or candidate[0] > best_tuple[0]:
                best_tuple = candidate
        return best_tuple

    def remove_car(self, idx):
        self.inventory.pop(idx)

## 3. Institutions and weekly market interaction

The institutional block is deliberately simple but meaningful.

An institution can:
- inspect cars with probability increasing in `institution_strength`;
- remove cars below a minimum quality threshold;
- impose a fine-like reduction in the seller's effective revenue.

This is not a full legal model. It is a compact representation of the idea that even modest inspections or standards can reshape seller incentives.

In [ ]:
def institutional_screen(price, car: Car, p: Params, rng: np.random.Generator):
    '''
    Returns:
        allowed_trade: bool
        net_price_received_by_seller: float
        inspected: bool
    '''
    inspect_prob = 0.05 + 0.80 * p.institution_strength
    inspected = rng.random() < inspect_prob

    if inspected and car.quality < p.minimum_quality_threshold:
        return False, 0.0, True

    # Fine / reduced effective revenue if a lemon is inspected
    if inspected and car.is_lemon:
        net_price = price * (1 - p.fine_multiplier * p.institution_strength)
    else:
        net_price = price

    return True, float(net_price), inspected


def simulate_week(p: Params, week_index: int, rng: np.random.Generator):
    buyers = [Buyer(p, rng) for _ in range(p.n_buyers)]
    sellers = [Seller(p, rng) for _ in range(p.n_sellers)]

    trades = 0
    lemon_sales = 0
    good_sales = 0
    inspected_sales = 0
    blocked_sales = 0
    sum_quality = 0.0
    prices_paid = []
    seller_revenues = []
    buyer_surpluses = []

    # Each buyer meets one seller. This captures the one-shot nature
    # emphasized in the Steinbeck interpretation.
    seller_indices = rng.integers(0, p.n_sellers, size=p.n_buyers)

    for buyer, s_idx in zip(buyers, seller_indices):
        seller = sellers[s_idx]
        if not seller.inventory:
            continue

        _, car_idx, car, quoted_price = seller.best_offer_for_buyer(buyer, p)
        perceived_q = buyer.perceived_quality(car, p, rng)
        reservation = buyer.reservation_price(perceived_q)

        # Purchase rule:
        # buyer buys if quoted price is below reservation value,
        # or if urgency is very high and price is still within the budget constraint.
        buy = (quoted_price <= reservation) or (buyer.urgency > 0.88 and quoted_price <= buyer.budget)

        if not buy:
            continue

        allowed_trade, net_revenue, inspected = institutional_screen(quoted_price, car, p, rng)

        if not allowed_trade:
            blocked_sales += 1
            seller.remove_car(car_idx)
            continue

        seller.remove_car(car_idx)
        trades += 1
        inspected_sales += int(inspected)
        lemon_sales += int(car.is_lemon)
        good_sales += int(not car.is_lemon)
        sum_quality += car.quality
        prices_paid.append(quoted_price)
        seller_revenues.append(net_revenue)
        buyer_surpluses.append(max(reservation - quoted_price, 0.0))

    avg_price = float(np.mean(prices_paid)) if prices_paid else 0.0
    avg_quality = float(sum_quality / trades) if trades else 0.0
    lemon_share = float(lemon_sales / trades) if trades else 0.0
    avg_buyer_surplus = float(np.mean(buyer_surpluses)) if buyer_surpluses else 0.0
    avg_seller_revenue = float(np.mean(seller_revenues)) if seller_revenues else 0.0

    return {
        "week": week_index,
        "trades": trades,
        "blocked_sales": blocked_sales,
        "lemon_sales": lemon_sales,
        "good_sales": good_sales,
        "lemon_share": lemon_share,
        "avg_price": avg_price,
        "avg_quality": avg_quality,
        "avg_buyer_surplus": avg_buyer_surplus,
        "avg_seller_revenue": avg_seller_revenue,
        "inspected_sales": inspected_sales,
    }

## 4. Running a full simulation

A simulation consists of repeated weekly markets over a fixed horizon.

The output is a panel with one row per week.  
This makes it possible to compute:
- weekly trajectories;
- annual averages;
- scenario comparisons.

In [ ]:
def run_simulation(p: Params):
    rng = np.random.default_rng(p.random_seed)
    rows = [simulate_week(p, t + 1, rng) for t in range(p.periods)]
    df = pd.DataFrame(rows)
    return df


def summarize_simulation(df: pd.DataFrame, scenario_name: str):
    return pd.DataFrame([{
        "scenario": scenario_name,
        "weeks": int(df["week"].max()),
        "total_trades": int(df["trades"].sum()),
        "mean_weekly_trades": float(df["trades"].mean()),
        "mean_lemon_share": float(df["lemon_share"].mean()),
        "mean_avg_quality": float(df["avg_quality"].mean()),
        "mean_avg_price": float(df["avg_price"].mean()),
        "mean_buyer_surplus": float(df["avg_buyer_surplus"].mean()),
        "mean_seller_revenue": float(df["avg_seller_revenue"].mean()),
        "blocked_sales_total": int(df["blocked_sales"].sum()),
    }])

## 5. Baseline run

The baseline is meant to capture a market with:
- substantial information asymmetry;
- strong urgency;
- low market literacy;
- appreciable seller opportunism;
- limited institutional correction.

In [ ]:
baseline_params = Params()
baseline_df = run_simulation(baseline_params)
baseline_summary = summarize_simulation(baseline_df, "baseline")

baseline_summary

In [ ]:
baseline_df.head(10)

## 6. Figures and tables for the paper

The next cell:

- saves figures to `figures/`;
- saves CSV outputs to `tables/`;
- writes LaTeX-ready `.tex` tables to `tables/`.

The LaTeX paper in this package expects these files to exist.  
So the notebook should be run before compiling the paper.

In [ ]:
def save_line_plot(df, x, y, title, ylabel, filename):
    fig, ax = plt.subplots()
    ax.plot(df[x], df[y], linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Week")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=180)
    plt.show()
    plt.close(fig)


def save_outputs(df: pd.DataFrame, summary_df: pd.DataFrame, prefix: str):
    # CSV
    df.to_csv(TAB_DIR / f"{prefix}_weekly.csv", index=False)
    summary_df.to_csv(TAB_DIR / f"{prefix}_summary.csv", index=False)

    # LaTeX tables
    summary_df.round(4).to_latex(
        TAB_DIR / f"{prefix}_summary.tex",
        index=False,
        escape=False,
        caption=f"Summary statistics for the {prefix.replace('_', ' ')} scenario.",
        label=f"tab:{prefix}_summary"
    )

    snapshot = df.loc[:, ["week", "trades", "lemon_share", "avg_quality", "avg_price", "avg_buyer_surplus"]].copy()
    snapshot = snapshot.round(4)
    snapshot.head(12).to_latex(
        TAB_DIR / f"{prefix}_first12weeks.tex",
        index=False,
        escape=False,
        caption=f"First 12 weeks of the {prefix.replace('_', ' ')} scenario.",
        label=f"tab:{prefix}_first12weeks"
    )

save_line_plot(baseline_df, "week", "lemon_share", "Baseline lemon share over time", "Lemon share", "baseline_lemon_share.png")
save_line_plot(baseline_df, "week", "avg_price", "Baseline average price over time", "Average price", "baseline_avg_price.png")
save_line_plot(baseline_df, "week", "avg_quality", "Baseline average quality over time", "Average quality", "baseline_avg_quality.png")

save_outputs(baseline_df, baseline_summary, "baseline")

## 7. Comparative scenarios

A central purpose of this notebook is to separate mechanisms.

We therefore compare four stylized scenarios:

1. **Akerlof-like information problem only**  
   High information asymmetry, but weaker Steinbeck amplification.
2. **Steinbeck distressed market**  
   High urgency, low literacy, higher opportunism and discrimination.
3. **Institutional correction**  
   Same distressed market, but stronger inspection and standards.
4. **More competitive market**  
   Same distressed market, but lower effective seller power.

In [ ]:
scenario_params = {
    "akerlof_only": Params(
        urgency_mean=0.35,
        literacy_mean=0.72,
        info_asymmetry=0.52,
        seller_opportunism=0.20,
        price_discrimination=0.10,
        institution_strength=0.12,
        competition_strength=0.35,
        random_seed=17
    ),
    "steinbeck_distress": Params(
        urgency_mean=0.82,
        literacy_mean=0.28,
        info_asymmetry=0.45,
        seller_opportunism=0.65,
        price_discrimination=0.65,
        institution_strength=0.05,
        competition_strength=0.12,
        random_seed=27
    ),
    "institutional_correction": Params(
        urgency_mean=0.82,
        literacy_mean=0.28,
        info_asymmetry=0.45,
        seller_opportunism=0.65,
        price_discrimination=0.65,
        institution_strength=0.70,
        competition_strength=0.12,
        random_seed=37
    ),
    "more_competition": Params(
        urgency_mean=0.82,
        literacy_mean=0.28,
        info_asymmetry=0.45,
        seller_opportunism=0.65,
        price_discrimination=0.65,
        institution_strength=0.05,
        competition_strength=0.75,
        random_seed=47
    ),
}

scenario_weekly = {}
scenario_summaries = []

for name, params in scenario_params.items():
    df = run_simulation(params)
    scenario_weekly[name] = df
    scenario_summaries.append(summarize_simulation(df, name))

comparison_summary = pd.concat(scenario_summaries, ignore_index=True)
comparison_summary

In [ ]:
# Save scenario comparison tables and a comparison figure

comparison_summary.round(4).to_csv(TAB_DIR / "scenario_comparison_summary.csv", index=False)
comparison_summary.round(4).to_latex(
    TAB_DIR / "scenario_comparison_summary.tex",
    index=False,
    escape=False,
    caption="Comparison of stylized scenarios.",
    label="tab:scenario_comparison_summary"
)

fig, ax = plt.subplots()

for name, df in scenario_weekly.items():
    ax.plot(df["week"], df["lemon_share"], label=name.replace("_", " "))

ax.set_title("Lemon share across scenarios")
ax.set_xlabel("Week")
ax.set_ylabel("Lemon share")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "scenario_comparison_lemon_share.png", dpi=180)
plt.show()
plt.close(fig)

## 8. Interactive slider panel

The next block provides an interactive panel.

To avoid accidental continuous recomputation, the notebook uses **`interact_manual`**, so a new simulation is run only when you click the button.

This is generally safer than automatic continuous updates when the number of buyers becomes large.

In [ ]:
def run_interactive_model(
    n_buyers=600,
    n_sellers=60,
    periods=52,
    cars_per_seller=4,
    share_good=0.35,
    info_asymmetry=0.42,
    urgency_mean=0.72,
    literacy_mean=0.38,
    seller_opportunism=0.55,
    price_discrimination=0.55,
    institution_strength=0.12,
    competition_strength=0.25,
    random_seed=7
):
    p = Params(
        n_buyers=n_buyers,
        n_sellers=n_sellers,
        periods=periods,
        cars_per_seller=cars_per_seller,
        share_good=share_good,
        info_asymmetry=info_asymmetry,
        urgency_mean=urgency_mean,
        literacy_mean=literacy_mean,
        seller_opportunism=seller_opportunism,
        price_discrimination=price_discrimination,
        institution_strength=institution_strength,
        competition_strength=competition_strength,
        random_seed=random_seed
    )

    df = run_simulation(p)
    summary = summarize_simulation(df, "interactive")

    display(Markdown("### Interactive summary"))
    display(summary.round(4))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(df["week"], df["lemon_share"], linewidth=2)
    axes[0].set_title("Lemon share")
    axes[0].set_xlabel("Week")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(df["week"], df["avg_price"], linewidth=2)
    axes[1].set_title("Average price")
    axes[1].set_xlabel("Week")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(df["week"], df["avg_quality"], linewidth=2)
    axes[2].set_title("Average quality")
    axes[2].set_xlabel("Week")
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    plt.close(fig)

    # Save the interactive run as well
    save_outputs(df, summary, "interactive")
    return df

interactive = widgets.interact_manual(
    run_interactive_model,
    n_buyers=widgets.IntSlider(value=600, min=100, max=4000, step=100, description="buyers"),
    n_sellers=widgets.IntSlider(value=60, min=10, max=300, step=10, description="sellers"),
    periods=widgets.IntSlider(value=52, min=4, max=104, step=4, description="weeks"),
    cars_per_seller=widgets.IntSlider(value=4, min=1, max=12, step=1, description="cars/seller"),
    share_good=widgets.FloatSlider(value=0.35, min=0.05, max=0.95, step=0.05, readout_format=".2f", description="share good"),
    info_asymmetry=widgets.FloatSlider(value=0.42, min=0.00, max=1.00, step=0.02, readout_format=".2f", description="info asym"),
    urgency_mean=widgets.FloatSlider(value=0.72, min=0.00, max=1.00, step=0.02, readout_format=".2f", description="urgency"),
    literacy_mean=widgets.FloatSlider(value=0.38, min=0.00, max=1.00, step=0.02, readout_format=".2f", description="literacy"),
    seller_opportunism=widgets.FloatSlider(value=0.55, min=0.00, max=1.20, step=0.02, readout_format=".2f", description="opportunism"),
    price_discrimination=widgets.FloatSlider(value=0.55, min=0.00, max=1.20, step=0.02, readout_format=".2f", description="discrimination"),
    institution_strength=widgets.FloatSlider(value=0.12, min=0.00, max=1.00, step=0.02, readout_format=".2f", description="institutions"),
    competition_strength=widgets.FloatSlider(value=0.25, min=0.00, max=1.00, step=0.02, readout_format=".2f", description="competition"),
    random_seed=widgets.IntSlider(value=7, min=1, max=999, step=1, description="seed"),
)

interactive.widget.manual_button.description = "Run simulation"
interactive.widget.manual_button.button_style = "primary"

## 9. Sensitivity experiment on institutions

The Steinbeck interpretation gives a central role to weak or absent corrective institutions.  
The next block runs a compact sensitivity experiment over different values of `institution_strength`.

In [ ]:
institution_grid = np.linspace(0, 1, 11)
institution_rows = []

for inst in institution_grid:
    p = Params(
        urgency_mean=0.82,
        literacy_mean=0.28,
        info_asymmetry=0.45,
        seller_opportunism=0.65,
        price_discrimination=0.65,
        institution_strength=float(inst),
        competition_strength=0.12,
        random_seed=123
    )
    df = run_simulation(p)
    institution_rows.append({
        "institution_strength": inst,
        "mean_lemon_share": df["lemon_share"].mean(),
        "mean_avg_quality": df["avg_quality"].mean(),
        "mean_avg_price": df["avg_price"].mean(),
        "mean_weekly_trades": df["trades"].mean()
    })

institution_sensitivity = pd.DataFrame(institution_rows)
institution_sensitivity.round(4)

In [ ]:
institution_sensitivity.round(4).to_csv(TAB_DIR / "institution_sensitivity.csv", index=False)
institution_sensitivity.round(4).to_latex(
    TAB_DIR / "institution_sensitivity.tex",
    index=False,
    escape=False,
    caption="Sensitivity of outcomes to institutional strength.",
    label="tab:institution_sensitivity"
)

fig, ax = plt.subplots()
ax.plot(institution_sensitivity["institution_strength"], institution_sensitivity["mean_lemon_share"], linewidth=2)
ax.set_title("Institutional strength and mean lemon share")
ax.set_xlabel("Institution strength")
ax.set_ylabel("Mean lemon share")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "institution_sensitivity_lemon_share.png", dpi=180)
plt.show()
plt.close(fig)

## 10. Interpretation notes

A few points are worth stressing.

### Akerlof-only case
If the main mechanism is informational, a high lemon share may emerge, but the market logic is still relatively impersonal.

### Steinbeck distress case
The model is designed to show that the market can become worse not only because of hidden quality, but because:
- buyers are in a hurry;
- buyers are weakly informed about market tactics;
- sellers exploit vulnerability directly;
- the interaction is effectively one-shot.

### Institutions and competition
Even without eliminating asymmetry, stronger institutions or stronger competition can reduce:
- lemon prevalence,
- average overcharging,
- the persistence of low-quality equilibria.

This is exactly the theoretical distinction motivating the model.

## 11. Next extensions you may want

Natural next steps include:

1. explicit spatial segmentation of local monopolies;
2. repeated seller reputation;
3. debt contracts and installment selling;
4. migration waves with changing buyer composition;
5. calibration to specific historical episodes.

The present notebook is intentionally transparent and expandable.